# 01 — EMG quality control and anomaly detection

**Phase B** (`docs/roadmap.md`). This notebook establishes the EMG signal-quality
layer that feeds Reborn's safety path, and evaluates the two complementary
detectors against *injected* faults:

1. **Deterministic QC** (`reborn.sensing.emg_qc`) — cheap, always-on, rule-based
   checks the safety layer trusts (dropout, saturation, clipping, amplitude range,
   baseline offset, mains interference). Named failure modes.
2. **Advisory anomaly detection** (`reborn.ml.anomaly`) — a one-class detector
   that flags *"this doesn't look like normal EMG"* without a per-mode rule.
   **Advisory only**: consumed via `reborn.decision.confidence_gate`, never wired
   to actuators, never overriding safety.

> **Data status: real.** This notebook now runs on downloaded Ninapro DB6
> recordings (`data/README.md`), not the synthetic smoke fixture it used before.
> Loading, preprocessing, and windowing all go through `reborn.data`, which calls
> `reborn.sensing` — so what is measured here is what the runtime does, not a
> parallel implementation of it.

**Two things this notebook must not do**, both of which it did implicitly while it
ran on synthetic data:

- Use the **default** QC thresholds. They are absolute, DB6 samples are of order
  1e-5 V, and the defaults reject ~100% of the dataset for reasons unrelated to
  signal quality. Thresholds come from
  `experiments/configs/ninapro_db6_qc.json`, derived by `reborn.data.qc_calibration`.
- Use the **default** corruption amplitudes. Also absolute; on DB6 they inject a
  fault ~50000x the signal, which every detector catches. Detection rates measured
  that way describe the injection, not the detector.

## Setup

Run with the interpreter that has scipy — `py -3.11` on this machine
(`docs/research/phase-b-plan.md` §10).

In [ ]:
import csv
import json
import time
from pathlib import Path

import numpy as np

from reborn.data.loaders import NinaproDB6Loader
from reborn.data.pipeline import PreprocessConfig, preprocess, window_recording
from reborn.data.qc_calibration import profile_amplitudes, suggest_corruption_kwargs
from reborn.ml.anomaly import AnomalyDetector
from reborn.sensing import corruption
from reborn.sensing.emg_qc import assess_quality_report
from reborn.sensing.features import extract_features

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads((REPO / "experiments" / "configs" / "ninapro_db6_qc.json").read_text())
QC_KWARGS = CONFIG["qc_kwargs"]
MONTAGE = tuple(CONFIG["channels"]["montage"])
PRE = CONFIG["preprocess"]

print("montage (raw DB6 columns):", MONTAGE)
print("QC thresholds:")
for key, value in QC_KWARGS.items():
    print(f"  {key:<18}{value:.4g}   [{CONFIG['derivation'][key]}]")

In [ ]:
config = PreprocessConfig(
    target_sample_rate=PRE["target_sample_rate_hz"],
    bandpass_hz=tuple(PRE["bandpass_hz"]),
    notch_hz=PRE["notch_hz"],
    window_ms=PRE["window_ms"],
    stride_ms=PRE["stride_ms"],
    pure_windows=PRE["pure_windows"],
    qc_kwargs=QC_KWARGS,
    # qc_channels stays None: the loader already narrowed the signal to MONTAGE,
    # so by this point channel 0 *is* raw column 0 and channel 1 *is* raw column
    # 10. Passing MONTAGE again here would index past the end of a 2-channel array.
    qc_channels=None,
)
print(f"window {config.window_samples} samples, stride {config.stride_samples}")
print("config fingerprint:", config.fingerprint())

## 1. EMG source — the real-data seam, now closed

The `load_emg_windows` stub this notebook used to carry is gone; recordings come
from `reborn.data.loaders.NinaproDB6Loader`.

**Scope knobs.** A full QC pass over one subject is ~136k windows and takes
minutes. Narrow `SESSIONS` while iterating; widen it for the run whose numbers you
keep. Raising `stride_ms` in the config above reduces the window count
proportionally — but it changes the fingerprint, so results at different strides
are not comparable and must not be mixed into one table.

In [ ]:
SUBJECTS = ["s01"]      # downloaded: s01, s02
SESSIONS = None         # None = every session for each subject

loader = NinaproDB6Loader(REPO / "data" / "ninapro_db6", channels=MONTAGE)
index = loader.index()

print(f"{len(index)} files, subjects: {loader.subjects()}")
print("sessions per subject:", sorted({session for _, session, _ in index}))

In [ ]:
# One recording, to see what the pipeline is actually handed.
sample_recording = next(loader.load(subjects=SUBJECTS[:1]))
prepared = preprocess(sample_recording, config)

print(f"{sample_recording.subject_id} / {sample_recording.session_id}")
print(
    f"  raw:      {sample_recording.n_samples} samples @ {sample_recording.sample_rate:.0f} Hz"
    f"  = {sample_recording.n_samples / sample_recording.sample_rate:.0f} s"
)
print(
    f"  prepared: {prepared.n_samples} samples @ {prepared.sample_rate:.0f} Hz,"
    f" {prepared.n_channels} channels"
)
print(f"  classes:  {np.unique(prepared.labels)}   (0 = rest)")
print(f"  rest:     {float(np.mean(prepared.labels == 0)):.1%} of samples")

## 2. QC rejection on real EMG — the first real result

The rejection rate is **not** bookkeeping. It is what the safety layer would have
done at runtime: these are the windows Reborn would refuse to act on. Reported per
session, because per-session variation is the thing worth seeing — an average over
sessions hides exactly the events that matter.

In [ ]:
rejection_rows = []
started = time.time()

print(f"{'subject':<10}{'session':<12}{'windows':>10}{'rejected':>10}{'rate':>9}   reasons")
print("-" * 78)
for recording in loader.load(subjects=SUBJECTS, sessions=SESSIONS):
    _, _, qc = window_recording(preprocess(recording, config), config)
    rejection_rows.append(
        {
            "subject": recording.subject_id,
            "session": recording.session_id,
            "windows": qc.total,
            "rejected": qc.rejected,
            "rejection_rate": qc.rejection_rate,
            **{f"reason_{name}": n for name, n in qc.rejected_by_reason.items()},
        }
    )
    print(
        f"{recording.subject_id:<10}{recording.session_id:<12}{qc.total:>10}"
        f"{qc.rejected:>10}{qc.rejection_rate:>8.2%}   {qc.rejected_by_reason}"
    )

total = sum(row["windows"] for row in rejection_rows)
rejected = sum(row["rejected"] for row in rejection_rows)
print("-" * 78)
print(f"{'ALL':<22}{total:>10}{rejected:>10}{rejected / total:>8.2%}")
print(f"\n({time.time() - started:.0f} s)")

In [ ]:
# Which sessions stand out? A session far above its neighbours is a signal-quality
# event worth naming in the paper, not noise to average away.
rates = np.array([row["rejection_rate"] for row in rejection_rows])
median = float(np.median(rates))

print(f"median {median:.2%}   min {rates.min():.2%}   max {rates.max():.2%}")
print(f"max/median: {rates.max() / median:.1f}x" if median else "median is zero")
print()
for row in sorted(rejection_rows, key=lambda r: -r["rejection_rate"])[:5]:
    print(f"  {row['subject']}/{row['session']}  {row['rejection_rate']:.2%}")

## 3. What the thresholds are made of

Re-derives the amplitude profile the thresholds came from, so the notebook shows
its own basis rather than trusting a committed JSON file. If these percentiles sit
far from the ones the config was built on, the config is stale and the rejection
rates above are measured against the wrong scale.

In [ ]:
profile = profile_amplitudes(loader.load(subjects=SUBJECTS[:1], sessions=SESSIONS))
summary = profile.summary()

print(f"{profile.n_windows} windows x {profile.n_channels} channels\n")
for stat in ("rms", "abs_mean", "abs_max"):
    line = "  ".join(f"{k}={v:.3g}" for k, v in summary[stat].items())
    print(f"{stat:<10}{line}")

print("\nthresholds these imply, vs. the committed config:")
from reborn.data.qc_calibration import suggest_qc_thresholds

for key, value in suggest_qc_thresholds(profile).items():
    drift = value / QC_KWARGS[key] if QC_KWARGS.get(key) else float("nan")
    print(f"  {key:<18}{value:.4g}   committed {QC_KWARGS[key]:.4g}   ratio {drift:.2f}x")

## 4. Deterministic QC vs. injected faults

Clean windows are corrupted one named mode at a time and re-checked. Two severity
settings run side by side, and the comparison is the point:

- **scaled** — amplitudes derived from this dataset (`suggest_corruption_kwargs`).
- **defaults** — `corruption.py`'s built-in values, sized for signals of order 1.

Expect the hard, nameable modes to be caught near-always. `noise_burst` is
deliberately **not** the deterministic layer's job — that is what the advisory
detector in §5 is for. If the two columns agree everywhere, the scaling is not
doing its job and the numbers describe the injection rather than the detector.

In [ ]:
# Clean windows to corrupt: gate wide open, so this is the signal as recorded
# rather than a view of it that the gate has already filtered.
OPEN_GATE = {
    "min_rms": 0.0,
    "max_rms": 1e18,
    "max_offset": 1e18,
    "saturation_limit": 1e18,
    "flatline_std": 0.0,
}
open_config = PreprocessConfig(
    target_sample_rate=config.target_sample_rate,
    bandpass_hz=config.bandpass_hz,
    notch_hz=config.notch_hz,
    window_ms=config.window_ms,
    stride_ms=config.stride_ms,
    pure_windows=config.pure_windows,
    qc_kwargs=OPEN_GATE,
)

one_session = next(loader.load(subjects=SUBJECTS[:1], sessions=SESSIONS))
clean_windows, _, _ = window_recording(preprocess(one_session, open_config), open_config)

N_FAULT_SAMPLE = 500
fault_sample = clean_windows[:N_FAULT_SAMPLE, :, 0]  # one channel, as the QC checks see it
print(
    f"{one_session.subject_id}/{one_session.session_id}: {clean_windows.shape[0]} windows,"
    f" using {fault_sample.shape[0]} for injection"
)

SCALED = suggest_corruption_kwargs(profile, severity=3.0)
print("\nscaled injection parameters:")
for mode, kwargs in SCALED.items():
    print(f"  {mode:<18}{kwargs}")

In [ ]:
baseline_valid = np.mean([assess_quality_report(w, **QC_KWARGS).valid for w in fault_sample])
print(f"uncorrupted sample passing the gate: {baseline_valid:.1%}\n")

fault_rows = []
print(f"{'mode':<18}{'scaled':>9}{'defaults':>10}   example failures (scaled)")
print("-" * 72)
for mode in corruption.FAULT_MODES:
    detected = {}
    example = ()
    for severity, kwargs in (("scaled", SCALED[mode]), ("defaults", {})):
        caught = 0
        for window in fault_sample:
            report = assess_quality_report(
                corruption.corrupt(window.copy(), mode, **kwargs), **QC_KWARGS
            )
            if not report.valid:
                caught += 1
                if severity == "scaled":
                    example = report.failures
        detected[severity] = caught / len(fault_sample)

    fault_rows.append(
        {
            "mode": mode,
            "detection_scaled": detected["scaled"],
            "detection_defaults": detected["defaults"],
            "injection": json.dumps(SCALED[mode]),
        }
    )
    print(f"{mode:<18}{detected['scaled']:>8.0%}{detected['defaults']:>10.0%}   {example}")

## 5. Advisory anomaly detector

Fit the one-class detector on features from clean **real** windows (RMS / MAV /
ZCR), then score held-out clean windows and each corruption. The clean flag rate
should sit near the `contamination` level; corrupted windows — especially
`noise_burst`, which §4 shows the deterministic layer skipping — should score
higher and flag more often.

This detector's output stays **advisory**. It lowers confidence through
`reborn.decision.confidence_gate`, and low confidence reduces assist, never
increases it (`docs/safety.md`).

In [ ]:
def feature_row(x):
    f = extract_features(x)
    return [f["rms"], f["mav"], f["zcr"]]


split = len(fault_sample) // 2
train = np.array([feature_row(w) for w in fault_sample[:split]])
held_out = fault_sample[split:]

detector = AnomalyDetector(contamination=0.025).fit(train)
clean_scores = [detector.score(feature_row(w)) for w in held_out]
clean_rate = float(np.mean([s.is_anomalous for s in clean_scores]))

print(f"Mahalanobis threshold: {detector.threshold:.3f}")
print(f"clean flag rate (held-out): {clean_rate:.1%}\n")

anomaly_rows = [
    {
        "mode": "clean",
        "flag_rate": clean_rate,
        "mean_score": float(np.mean([s.score for s in clean_scores])),
    }
]

print(f"{'mode':<18}{'flag rate':>11}{'mean score':>13}")
print("-" * 42)
print(f"{'clean':<18}{clean_rate:>10.1%}{anomaly_rows[0]['mean_score']:>13.2f}")
for mode in corruption.FAULT_MODES:
    scores = [
        detector.score(feature_row(corruption.corrupt(w.copy(), mode, **SCALED[mode])))
        for w in held_out
    ]
    rate = float(np.mean([s.is_anomalous for s in scores]))
    mean_score = float(np.mean([s.score for s in scores]))
    anomaly_rows.append({"mode": mode, "flag_rate": rate, "mean_score": mean_score})
    print(f"{mode:<18}{rate:>10.1%}{mean_score:>13.2f}")

## 6. Artifacts

The notebook does not *hold* the result — it writes one. Figures are rendered from
these CSVs by a separate script, so every number in the paper traces back to a
file and a config fingerprint rather than to a live kernel
(`docs/research/phase-b-plan.md` §8).

`experiments/results/` is git-ignored; the small aggregated tables that reach the
manuscript get copied into `papers/drift_personalization/results/` deliberately.

In [ ]:
def write_csv(path, rows):
    if not rows:
        print(f"skipped {path.name}: nothing to write")
        return
    fields = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f"wrote {path.relative_to(REPO)}  ({len(rows)} rows)")


stamp = config.fingerprint()
write_csv(RESULTS / f"nb01_qc_rejection_{stamp}.csv", rejection_rows)
write_csv(RESULTS / f"nb01_fault_detection_{stamp}.csv", fault_rows)
write_csv(RESULTS / f"nb01_anomaly_{stamp}.csv", anomaly_rows)

(RESULTS / f"nb01_run_{stamp}.json").write_text(
    json.dumps(
        {
            "config_fingerprint": stamp,
            "dataset": "ninapro_db6",
            "subjects": SUBJECTS,
            "sessions": SESSIONS or "all",
            "montage_raw_columns": list(MONTAGE),
            "qc_kwargs": QC_KWARGS,
            "corruption_severity": 3.0,
            "fault_sample_windows": int(fault_sample.shape[0]),
            "windows_assessed": int(total),
        },
        indent=2,
    )
)
print(f"wrote experiments/results/nb01_run_{stamp}.json")

## 7. Reading & next steps

**Two layers, on purpose.** The deterministic checks give the safety path cheap,
explainable, named guarantees; the advisory detector adds coverage for
degradations that no single threshold names. The detector's output stays advisory
— it lowers confidence through `reborn.decision.confidence_gate`, and **low
confidence reduces assist, never increases it** (`docs/safety.md`,
`docs/research/research-context.md` §5.3).

### Notes from this run

> _Written against the outputs above, not in advance. Each note should say what
> the number is and which design decision it changes; a number that changes
> nothing does not need a note (`docs/experiments.md`: "If an experiment does not
> change a design decision, it is not needed")._

- **Rejection rate (§2)** — TBD
- **Per-session spread (§2)** — TBD
- **Threshold drift check (§3)** — TBD
- **Fault detection, scaled vs. defaults (§4)** — TBD
- **Advisory detector coverage (§5)** — TBD

### Next

1. Repeat on the held-out subject (`SUBJECTS = ["s02"]`) and confirm the
   thresholds transfer — they were derived on s01, so s01's rejection rate is not
   an independent check of them.
2. Characterise cross-session drift of the clean-signal statistics (the §3 profile,
   per session) — the bridge into `03_drift_fewshot`.
3. Plots — amplitude traces per fault, score distributions — as a separate script
   reading the CSVs from §6. This notebook stays text-only so it runs anywhere.